# EPFL Hackathon 2026 - Data Preprocesing

In [24]:
import numpy as np
import pandas as pd

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer.primitives import Estimator
from qiskit_aer import Aer
from qiskit import transpile
from scipy.optimize import minimize

In [25]:
# 0) LOAD + SMALL BATCH
df = pd.read_csv("underwriting_dataset.csv")
df.columns = df.columns.str.strip()

# IMPORTANT: QAOA on laptop => keep n small (e.g., 20-24)
n_use = 24  # change to 20 if slow
# pick the most "useful" cases to make the demo non-trivial:
# here: highest v_i (you can use p_fraud or severity_L too)
df_small = df.sort_values("v_i", ascending=False).head(n_use).reset_index(drop=True)

v = df_small["v_i"].astype(float).to_numpy()
n = len(df_small)

# choose exactly K cases
K = min(8, n)
lam = 1.0

# penalty should dominate typical gains; start robustly
rho = 50.0 * max(1.0, np.max(np.abs(v)))  # scale with data magnitude

In [26]:
# 1) BUILD NETWORK W 

W = np.zeros((n, n), dtype=float)

def add_links(col, weight):
    vals = df_small[col].to_numpy()
    for i in range(n):
        for j in range(i + 1, n):
            if vals[i] == vals[j]:
                W[i, j] += weight
                W[j, i] += weight

# columns exist in your dataset
add_links("garage_id",      1.0)
add_links("device_id",      1.0)
add_links("broker_id",      0.5)
add_links("ring_id_latent", 2.0)
add_links("region_id",      0.2)

# sparsify
threshold = 0.5
W[W < threshold] = 0.0

In [27]:
# 2) BUILD QUBO 
qp = QuadraticProgram("fraud_alert_prioritization")

for i in range(n):
    qp.binary_var(f"x_{i}")

linear = {}
quadratic = {}

def add_q(u, vname, c):
    key = tuple(sorted((u, vname)))
    quadratic[key] = quadratic.get(key, 0.0) + float(c)

# (A) -sum v_i x_i
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - float(v[i])

# (B) -lam * sum_{i<j} W_ij x_i x_j
for i in range(n):
    for j in range(i + 1, n):
        if W[i, j] != 0.0:
            add_q(f"x_{i}", f"x_{j}", -lam * float(W[i, j]))

# (C) rho*(sum x - K)^2
# Expand: rho*(sum x)^2 - 2*rho*K*(sum x) + const
# and (sum x)^2 = sum x_i + 2*sum_{i<j} x_i x_j
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) + rho
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - 2.0 * rho * K

for i in range(n):
    for j in range(i + 1, n):
        add_q(f"x_{i}", f"x_{j}", 2.0 * rho)

qp.minimize(linear=linear, quadratic=quadratic)
qubo = QuadraticProgramToQubo().convert(qp)

In [28]:
# 3) QAOA (ROBUST: Estimator + COBYLA)
H, offset = qubo.to_ising()

reps = 2
ansatz = QAOAAnsatz(cost_operator=H, reps=reps)

estimator = Estimator()

def energy(theta):
    job = estimator.run([ansatz], [H], [theta])
    return float(job.result().values[0])

theta0 = np.random.default_rng(0).uniform(0, 2*np.pi, size=ansatz.num_parameters)
opt = minimize(energy, theta0, method="COBYLA", options={"maxiter": 200})
theta_star = opt.x

# sample optimized circuit
qc = ansatz.assign_parameters(theta_star)
qc.measure_all()

backend = Aer.get_backend("aer_simulator")
tqc = transpile(qc, backend)
counts = backend.run(tqc, shots=5000).result().get_counts()

In [29]:
# 4) DECODE + pick BEST FEASIBLE 
var_names = [var.name for var in qubo.variables]  # should be x_0..x_{n-1}

def decode(bitstring):
    bits = np.array([int(b) for b in bitstring[::-1]], dtype=int)  # reverse to qubit order
    if len(bits) != len(var_names):
        raise ValueError(f"len(bits)={len(bits)} != num_vars={len(var_names)}")
    assign = dict(zip(var_names, bits))
    x = np.array([assign[f"x_{i}"] for i in range(n)], dtype=int)
    return x

def score_business(x):
    # value + synergy (without penalty), for reporting
    val = float(np.sum(v * x))
    syn = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            syn += W[i, j] * x[i] * x[j]
    return val + lam * float(syn), val, float(syn)

best_feasible = None
best_feasible_score = -1e18

# search feasible bitstrings among measured samples
for bitstr, ct in counts.items():
    x = decode(bitstr)
    if int(x.sum()) == K:
        total, val, syn = score_business(x)
        if total > best_feasible_score:
            best_feasible_score = total
            best_feasible = (bitstr, x, val, syn)

# fallback: if no feasible found (rare when rho is large), take most frequent
if best_feasible is None:
    best_bitstr = max(counts, key=counts.get)
    x = decode(best_bitstr)
    total, val, syn = score_business(x)
else:
    best_bitstr, x, val, syn = best_feasible
    total = val + lam * syn

selected_idx = np.where(x == 1)[0]

In [30]:
# 5) REPORT
print("Used n =", n, "K =", K, "rho =", rho, "reps =", reps)
print("Best bitstring:", best_bitstr)
print("Selected count:", int(x.sum()), "(target K =", K, ")")
print("Selected indices (within df_small):", selected_idx.tolist())
print("Sum individual value:", val)
print("Network synergy term:", lam * syn)
print("Total (value + synergy):", total)

# show selected rows 
display(df_small.loc[selected_idx].reset_index(drop=True))

Used n = 24 K = 8 rho = 30755.416807343612 reps = 2
Best bitstring: 000000011010000100011011
Selected count: 8 (target K = 8 )
Selected indices (within df_small): [0, 1, 3, 4, 8, 13, 15, 16]
Sum individual value: 1548.2376950564505
Network synergy term: 26.799999999999997
Total (value + synergy): 1575.0376950564505


,case_id,claim_type,region_id,claim_amount,policy_age_days,prior_claims,complexity,review_minutes,broker_id,garage_id,device_id,entity_risk_score,p_fraud,ring_id_latent,severity_L,review_cost,v_i
0,10,home,1,18072.525159,3501,0,0.871113,50,0,3,2,0.228137,0.167283,1,7229.010064,50.0,615.108336
1,44,home,3,13379.388121,2377,0,0.845113,49,0,10,35,0.106681,0.134868,1,5351.755249,49.0,347.979272
2,34,home,1,11388.205003,943,1,0.954102,61,7,10,10,-0.109058,0.145228,1,4555.282001,61.0,302.854042
3,21,home,2,7205.645085,1703,2,0.300000,46,8,11,20,-0.189566,0.151894,2,2882.258034,46.0,194.789143
4,29,auto,0,5745.863440,24,1,0.481235,44,2,10,9,0.010882,0.140320,0,1723.759032,44.0,89.033034
5,5,home,2,4912.439475,733,1,1.010721,63,4,1,22,-0.038604,0.063981,0,1964.975790,63.0,6.146539
6,13,auto,3,4501.723507,2087,0,0.610656,41,0,10,4,0.034484,0.051437,0,1350.517052,41.0,-2.793055
7,18,auto,2,4635.492242,3505,1,0.896200,59,8,9,25,0.004896,0.070759,0,1390.647673,59.0,-4.879617
